# Analise exploratória dos microdados do Enem 2023

In [7]:
import pandas as pd
import numpy as np

In [ ]:
file_path = '../data/raw/MICRODADOS_ENEM_2023.csv'
# [Acessar fonte](https://download.inep.gov.br/microdados/microdados_enem_2023.zip) 

df_amostra = pd.read_csv(file_path, sep=';', encoding='latin1')
# aproximadamente 2m30s para rodar o dataset inteiro

# df_amostra = pd.read_csv(file_path, sep=';', encoding='latin1', nrow=1000)

KeyboardInterrupt: 

In [9]:
df_amostra.head(20);

## Especificações do dataset
- Estudar dimensionalidade do dataset
- Estudar preliminarmente a natureza das features
- Buscar por valores vazios

In [10]:
print(f"O dataset possui {df_amostra.shape[0]} linhas e {df_amostra.shape[1]} colunas.\n")

O dataset possui 3933955 linhas e 76 colunas.



In [11]:
print("Visualizando os tipos de dados de cada variável:")
display(df_amostra.dtypes)

Visualizando os tipos de dados de cada variável:


NU_INSCRICAO       int64
NU_ANO             int64
TP_FAIXA_ETARIA    int64
TP_SEXO              str
TP_ESTADO_CIVIL    int64
                   ...  
Q021                 str
Q022                 str
Q023                 str
Q024                 str
Q025                 str
Length: 76, dtype: object

In [12]:
print("Visualizando especificações numéricas do dataset:")
display(df_amostra.info)

Visualizando especificações numéricas do dataset:


<bound method DataFrame.info of          NU_INSCRICAO  NU_ANO  TP_FAIXA_ETARIA TP_SEXO  TP_ESTADO_CIVIL  \
0        210059085136    2023               14       M                2   
1        210059527735    2023               12       M                2   
2        210061103945    2023                6       F                1   
3        210060214087    2023                2       F                1   
4        210059980948    2023                3       F                1   
...               ...     ...              ...     ...              ...   
3933950  210061959676    2023               12       M                1   
3933951  210061950911    2023                1       F                1   
3933952  210061965966    2023                3       F                1   
3933953  210061932304    2023                2       M                1   
3933954  210058924455    2023                3       F                1   

         TP_COR_RACA  TP_NACIONALIDADE  TP_ST_CONCLUSAO  TP_ANO_CON

### Construção do Dicionário de Dados

Será conduzido uma analise acerda das características de cada feature do dataset. 

O arquivo `resources/Dicionário_Microdados_Enem_2023.xlsx` representa o dicionário de dados oficial publicado juntamente ao dataset. A contrução do dicionário de dados neste nootebook será guiado por meio do dicionário oficial.

In [15]:
dados_dicionario = []

for coluna in df_amostra.columns:
    tipo = df_amostra[coluna].dtype
    valores_unicos = df_amostra[coluna].dropna().unique()
    
    # Pega até 5 exemplos de valores distintos
    exemplos = [str(v) for v in valores_unicos[:5]]
    
    dados_dicionario.append({
        'Nome da Variável': coluna,
        'Tipo de Dado': str(tipo),
        'Qtd Nulos': df_amostra[coluna].isnull().sum(),
        'Total Únicos': len(valores_unicos),
        'Exemplos Práticos': " | ".join(exemplos)
    })

df_dicionario = pd.DataFrame(dados_dicionario)

pd.set_option('display.max_rows', 150)
df_dicionario

,Nome da Variável,Tipo de Dado,Qtd Nulos,Total Únicos,Exemplos Práticos
0,NU_INSCRICAO,int64,0,3933955,210059085136 | 210059527735 | 210061103945 | 2...
1,NU_ANO,int64,0,1,2023
2,TP_FAIXA_ETARIA,int64,0,20,14 | 12 | 6 | 2 | 3
3,TP_SEXO,str,0,2,M | F
4,TP_ESTADO_CIVIL,int64,0,5,2 | 1 | 0 | 3 | 4
5,TP_COR_RACA,int64,0,6,1 | 3 | 2 | 0 | 5
6,TP_NACIONALIDADE,int64,0,5,1 | 0 | 4 | 2 | 3
7,TP_ST_CONCLUSAO,int64,0,4,1 | 2 | 3 | 4
8,TP_ANO_CONCLUIU,int64,0,18,17 | 16 | 0 | 12 | 1
9,TP_ESCOLA,int64,0,3,1 | 2 | 3


Após comparar o dicionário acima com o dicionário oficial, não foi encontrado divergências.

### Classificação das Variáveis (Tipologia dos Dados)

Para justificar nossa análise e facilitar as manipulações futuras, podemos separar as variáveis em grandes grupos de natureza de dados, baseando-se em suas descrições no Dicionário Oficial:

1. **Dados Categóricos (Qualitativos):** 
   - São variáveis que representam características, grupos ou códigos. 
   - **Justificativa:** Muitos deles são números (ex: `TP_SEXO`, `CO_MUNICIPIO_ESC`, questões do tipo `Q001` a `Q025`), mas não têm valor quantitativo (não faz sentido calcular a média de códigos numéricos de município). Eles representam categorias (1 para Branca, 2 para Preta, 1 para Masculino, letras para classe social).
   
2. **Dados Numéricos (Quantitativos Contínuos):**
   - São os valores que representam medidas ou grandezas contínuas.
   - **Justificativa:** No Enem, referem-se estritamente ao desempenho escolar do candidato (Notas das provas objetivas e competências da redação: `NU_NOTA_CN`, `NU_NOTA_REDACAO`, etc). Aqui faz total sentido aplicar cálculos estatísticos tradicionais (média, variação, etc).

3. **Dados Textuais ("Escritos" ou Strings Literais):**
   - Dados que representam texto descritivo.
   - **Justificativa:** Nomes dos municípios por extenso e siglas das Unidades Federativas (`NO_MUNICIPIO`, `SG_UF`). Também as cadeias de respostas e gabaritos (`TX_RESPOSTAS`, `TX_GABARITO`), que são agrupamentos de caracteres muito únicos para tratamos como "categorias agrupáveis".
   
4. **Identificadores (Chave):**
   - Servem apenas como indexador (ex: `NU_INSCRICAO`). Não entra em análises diretas.

In [ ]:
# Definindo listas de colunas para cada categoria baseada no Dicionário Oficial
colunas_identificadoras = ['NU_INSCRICAO'] # Apenas Chave de Identificação Primária

# Dados Numéricos (Notas do candidato - grandezas que podem ser somadas, calculadas: média, desvio padrão)
colunas_numericas = [
    'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 
    'NU_NOTA_COMP1', 'NU_NOTA_COMP2', 'NU_NOTA_COMP3', 'NU_NOTA_COMP4', 
    'NU_NOTA_COMP5', 'NU_NOTA_REDACAO'
]

# Dados Escritos / Textuais Liberais (Como nomes por extenso, siglas literais, e os arrays literais de respostas e gabaritos)
colunas_textuais = [
    'NO_MUNICIPIO_ESC', 'SG_UF_ESC',
    'NO_MUNICIPIO_PROVA', 'SG_UF_PROVA',
    'TX_RESPOSTAS_CN', 'TX_RESPOSTAS_CH', 'TX_RESPOSTAS_LC', 'TX_RESPOSTAS_MT',
    'TX_GABARITO_CN', 'TX_GABARITO_CH', 'TX_GABARITO_LC', 'TX_GABARITO_MT'
]

# Dados Categóricos (O restante: opções numéricas descritivas dos manuais, tipo da prova, 
# situação do aluno, e o questionário Q001 a Q025)
colunas_categoricas = [
    col for col in df_amostra.columns 
    if col not in colunas_numericas and col not in colunas_textuais and col not in colunas_identificadoras
]

# Mostrando o agrupamento
print(f"Colunas Numéricas/Quantitativas: {len(colunas_numericas)}")
print(f"Colunas Textuais/Escritas: {len(colunas_textuais)}")
print(f"Colunas Categóricas/Qualitativas: {len(colunas_categoricas)}")
print(f"Colunas de Chave Primária: {len(colunas_identificadoras)}")

# Visualização de Categorias vs Tipos de Variáveis Extraídas
agrupamentos = {
    'Tipo': ['Numéricos', 'Textuais', 'Categóricos'],
    'Exemplos Representativos': [
        str(colunas_numericas[:5]),
        str(colunas_textuais[:5]),
        str(colunas_categoricas[:5])
    ]
}

pd.DataFrame(agrupamentos)